## Create the Environment

### Implement a movable square controlled by WASD, with acceleration and inertia.

In [14]:
import pygame
import numpy as np
from dataclasses import dataclass
from typing import Tuple
import sys

pygame.init()

WINDOW_W, WINDOW_H = 800, 600
FPS = 60
FIELD_W, FIELD_H = 700, 500
FIELD_OFFSET = ((WINDOW_W - FIELD_W) // 2, (WINDOW_H - FIELD_H) // 2)

COLOR_BG       = (20,  20,  30)
COLOR_GRID     = (40,  40,  55)
COLOR_PLAYER   = (80,  180, 255)
COLOR_BORDER   = (100, 100, 120)
COLOR_HUD      = (200, 200, 210)

### Define the Player Dataclass

Core attributes:

- **pos**: floating-point position `[x, y]` in field coordinates
- **vel**: velocity vector (px/frame), persists with inertia
- **accel**: base acceleration magnitude from key input
- **friction**: velocity decay per frame, simulates inertia (0.90 = 90% retained)
- **max_speed**: speed cap

The `rect` property converts field coordinates to screen coordinates for pygame rendering.

In [15]:
@dataclass
class Player:
    pos: np.ndarray
    vel: np.ndarray
    size: int      = 20
    color: Tuple   = COLOR_PLAYER
    accel: float   = 0.8
    friction: float = 0.90
    max_speed: float = 6.0

    hp: float = 100.0
    max_hp: float = 100.0
    attack_range: float = 45.0
    attack_damage: float = 10.0
    attack_cooldown: int = 30
    attack_cooldown_remaining: int = 0

    @property
    def rect(self) -> pygame.Rect:
        screen_x = int(FIELD_OFFSET[0] + self.pos[0] - self.size / 2)
        screen_y = int(FIELD_OFFSET[1] + self.pos[1] - self.size / 2)
        return pygame.Rect(screen_x, screen_y, self.size, self.size)

    @property
    def center(self) -> np.ndarray:
        return self.pos.copy()

    def is_dead(self) -> bool:
        return self.hp <= 0.0

    def take_damage(self, dmg: float):
        self.hp = max(0.0, self.hp - dmg)

p = Player(
    pos=np.array([FIELD_W / 2, FIELD_H / 2], dtype=np.float32),
    vel=np.array([0.0, 0.0], dtype=np.float32),
)
print(f"Pos: {p.pos}, Vel: {p.vel}, Rect: {p.rect}")


Pos: [350. 250.], Vel: [0. 0.], Rect: <rect(390, 290, 20, 20)>


### WASD Input to Acceleration Vector

Key mapping:

| Key | Acceleration Direction |
|-----|------------------------|
| W   | Up    [0, -1]          |
| S   | Down  [0, +1]          |
| A   | Left  [-1, 0]          |
| D   | Right [+1, 0]          |

Multiple keys sum their direction vectors, then the result is normalized so diagonal movement is not faster than cardinal.

In [16]:
KEY_MAP = {
    pygame.K_w: np.array([ 0.0, -1.0]),
    pygame.K_s: np.array([ 0.0,  1.0]),
    pygame.K_a: np.array([-1.0,  0.0]),
    pygame.K_d: np.array([ 1.0,  0.0]),
}

def get_accel_from_keys(keys, accel_magnitude: float = 0.8) -> np.ndarray:
    accel = np.array([0.0, 0.0], dtype=np.float32)
    for key, direction in KEY_MAP.items():
        if keys[key]:
            accel += direction
    norm = np.linalg.norm(accel)
    if norm > 1e-6:
        accel = accel / norm * accel_magnitude
    return accel

class FakeKeys:
    def __init__(self, pressed):
        self._pressed = set(pressed)
    def __getitem__(self, key):
        return key in self._pressed

fake = FakeKeys([pygame.K_w, pygame.K_d])
a = get_accel_from_keys(fake)
print(f"W+D accel: {a}, magnitude: {np.linalg.norm(a):.3f}  (expect ~0.8)")

W+D accel: [ 0.56568545 -0.56568545], magnitude: 0.800  (expect ~0.8)


### Physics Update & Boundary Constraints

Per-frame update flow:

1. Get acceleration vector from key state
2. Accumulate velocity: `vel += accel`
3. Apply friction: `vel *= friction`
4. Clamp speed to max_speed
5. Update position: `pos += vel`
6. Boundary collision: bounce off field edges with 50% velocity decay

In [17]:
def physics_update_entity(
    pos: np.ndarray, vel: np.ndarray, accel: np.ndarray,
    friction: float, max_speed: float, size: int,
    dt: float = 1.0
) -> tuple:
    """纯物理更新，返回 (new_pos, new_vel)"""
    vel = vel + accel * dt
    vel *= friction

    speed = np.linalg.norm(vel)
    if speed > max_speed:
        vel = vel / speed * max_speed

    pos = pos + vel * dt

    half = size / 2
    for axis in (0, 1):
        limit = FIELD_W if axis == 0 else FIELD_H
        if pos[axis] - half < 0:
            pos[axis] = half
            vel[axis] = abs(vel[axis]) * 0.5
        elif pos[axis] + half > limit:
            pos[axis] = limit - half
            vel[axis] = -abs(vel[axis]) * 0.5

    return pos, vel


def player_accel_from_keys(keys, accel_magnitude: float = 0.8) -> np.ndarray:
    """人类玩家按键 -> 加速度向量"""
    accel = np.array([0.0, 0.0], dtype=np.float32)
    for key, direction in KEY_MAP.items():
        if keys[key]:
            accel += direction
    norm = np.linalg.norm(accel)
    if norm > 1e-6:
        accel = accel / norm * accel_magnitude
    return accel


MOVE_VECTORS = {
    0: (0.0, 0.0),   1: (0.0, -1.0),  2: (-1.0, 0.0),
    3: (0.0, 1.0),   4: (1.0, 0.0),   5: (1.0, -1.0),
    6: (1.0, 1.0),   7: (-1.0, 1.0),  8: (-1.0, -1.0),
}

def action_to_accel(move_idx: int, accel_magnitude: float = 0.8) -> np.ndarray:
    """RL 动作的移动索引 -> 加速度向量

    move_idx: 0~8, 对应 MOVE_ACTIONS = ["STOP","W","A","S","D","WA","WD","SA","SD"]
    """
    dx, dy = MOVE_VECTORS[move_idx]
    v = np.array([dx, dy], dtype=np.float32)
    norm = np.linalg.norm(v)
    if norm > 1e-6:
        v = v / norm * accel_magnitude
    return v


### Render Function

Per-frame draw:

1. Clear screen with dark background
2. 50px grid lines (visual positioning aid)
3. Field border rectangle
4. Player square (blue)
5. Velocity direction indicator line (yellow, 10x scale)
6. HUD text (position, speed, controls)

In [18]:
def init_pygame_window() -> pygame.Surface:
    screen = pygame.display.set_mode((WINDOW_W, WINDOW_H))
    pygame.display.set_caption("RL Arena - Player Movement Test")
    return screen

def render_frame(screen: pygame.Surface, player: Player, font: pygame.font.Font) -> None:
    screen.fill(COLOR_BG)

    grid_spacing = 50
    for x in range(0, FIELD_W + 1, grid_spacing):
        sx = FIELD_OFFSET[0] + x
        pygame.draw.line(screen, COLOR_GRID, (sx, FIELD_OFFSET[1]), (sx, FIELD_OFFSET[1] + FIELD_H), 1)
    for y in range(0, FIELD_H + 1, grid_spacing):
        sy = FIELD_OFFSET[1] + y
        pygame.draw.line(screen, COLOR_GRID, (FIELD_OFFSET[0], sy), (FIELD_OFFSET[0] + FIELD_W, sy), 1)

    border_rect = pygame.Rect(FIELD_OFFSET[0], FIELD_OFFSET[1], FIELD_W, FIELD_H)
    pygame.draw.rect(screen, COLOR_BORDER, border_rect, 2)

    pygame.draw.rect(screen, player.color, player.rect)

    cx = int(FIELD_OFFSET[0] + player.pos[0])
    cy = int(FIELD_OFFSET[1] + player.pos[1])
    vx = int(player.vel[0] * 10)
    vy = int(player.vel[1] * 10)
    pygame.draw.line(screen, (255, 255, 100), (cx, cy), (cx + vx, cy + vy), 2)

    hud_lines = [
        f"Pos: ({player.pos[0]:.1f}, {player.pos[1]:.1f})",
        f"Vel: ({player.vel[0]:.2f}, {player.vel[1]:.2f})  |speed|: {np.linalg.norm(player.vel):.2f}",
        "WASD to move | ESC to quit",
    ]
    for i, line in enumerate(hud_lines):
        text = font.render(line, True, COLOR_HUD)
        screen.blit(text, (10, 10 + i * 22))

    pygame.display.flip()

### Run Test

Tie everything together: `init -> create player -> loop { events -> physics -> render -> tick }`

In [ ]:
screen = init_pygame_window()
clock = pygame.time.Clock()
font = pygame.font.SysFont("consolas", 16)

player = Player(
    pos=np.array([FIELD_W / 2, FIELD_H / 2], dtype=np.float32),
    vel=np.array([0.0, 0.0], dtype=np.float32),
)

# try:
#     running = True
#     while running:
#         for event in pygame.event.get():
#             if event.type == pygame.QUIT:
#                 running = False
#             if event.type == pygame.KEYDOWN and event.key == pygame.K_ESCAPE:
#                 running = False

#         keys = pygame.key.get_pressed()
#         accel = player_accel_from_keys(keys, player.accel)
#         player.pos, player.vel = physics_update_entity(
#             player.pos, player.vel, accel,
#             player.friction, player.max_speed, player.size,
#         )
#         render_frame(screen, player, font)
#         clock.tick(FPS)
# finally:
#     pygame.quit()

## 规则 Bot 对手

Bot 使用脚本化策略：

1. **追逐**：每帧计算 Bot 到目标的距离，若 > 攻击距离，则向目标方向移动
2. **攻击**：距离 <= 攻击距离时自动攻击（有冷却）
3. **决策频率**：每帧都重新计算方向（模拟实时反应）

Bot 的属性：
- hp, max_hp
- attack_range（攻击判定距离）
- attack_cooldown（攻击间隔，帧数）
- attack_damage（单次伤害）
- attack_cooldown_remaining（当前冷却剩余帧数）


In [20]:
@dataclass
class Bot:
    pos: np.ndarray          # [x, y]
    vel: np.ndarray          # [vx, vy]
    size: int = 20
    color: tuple = (255, 80, 80)  # 红色
    accel: float = 0.6
    friction: float = 0.90
    max_speed: float = 5.0

    hp: float = 100.0
    max_hp: float = 100.0
    attack_range: float = 30.0     # 攻击判定距离（含双方半径）
    attack_damage: float = 8.0
    attack_cooldown: int = 30      # 30 帧 = 0.5s @60fps
    attack_cooldown_remaining: int = 0

    def get_action(self, target_pos: np.ndarray) -> np.ndarray:
        """返回加速度方向向量（归一化），用于 physics_update"""
        direction = target_pos - self.pos
        dist = np.linalg.norm(direction)
        if dist < 1e-6:
            return np.array([0.0, 0.0], dtype=np.float32)
        return direction / dist * self.accel

    def should_attack(self, target_pos: np.ndarray) -> bool:
        """距离 <= attack_range 且冷却完毕时返回 True"""
        dist = np.linalg.norm(target_pos - self.pos)
        return dist <= self.attack_range and self.attack_cooldown_remaining == 0

    def step_cooldown(self):
        """每帧调用，递减冷却"""
        if self.attack_cooldown_remaining > 0:
            self.attack_cooldown_remaining -= 1

    def take_damage(self, dmg: float):
        self.hp = max(0.0, self.hp - dmg)

    def is_dead(self) -> bool:
        return self.hp <= 0.0

    @property
    def rect(self) -> pygame.Rect:
        screen_x = int(FIELD_OFFSET[0] + self.pos[0] - self.size / 2)
        screen_y = int(FIELD_OFFSET[1] + self.pos[1] - self.size / 2)
        return pygame.Rect(screen_x, screen_y, self.size, self.size)

    @property
    def center(self) -> np.ndarray:
        return self.pos.copy()

## 战斗系统

### 攻击判定

每帧 step 中：
1. 玩家攻击：若 skill 动作指示释放且距离 <= attack_range 且冷却完毕 -> 对 Bot 造成伤害，重置冷却
2. Bot 攻击：`should_attack()` 返回 True -> 对玩家造成伤害，设置冷却
3. 双方独立维护冷却计数器

### 死亡 & 超时

- 任一方 `hp <= 0` -> `done = True`，存活方获胜
- `step_count >= max_steps` -> `done = True`，按剩余血量判定胜负
- 若双方同时死亡 -> 平局

### 速度惩罚

攻击方在攻击瞬间施加短暂速度衰减（50%），模拟"出招硬直"，防止无限追击连砍。


In [21]:
def try_attack(
    attacker_pos: np.ndarray, defender,
    attack_range: float, attack_damage: float,
    cooldown_remaining: int, cooldown_max: int,
    attacker_vel: np.ndarray, speed_penalty: float = 0.5,
) -> tuple:
    """
    尝试攻击。
    返回: (attack_triggered, new_cooldown, actual_damage, new_attacker_vel)
    """
    if cooldown_remaining > 0:
        return False, cooldown_remaining, 0.0, attacker_vel

    dist = np.linalg.norm(defender.center - attacker_pos)
    if dist > attack_range:
        return False, cooldown_remaining, 0.0, attacker_vel

    defender.hp = max(0.0, defender.hp - attack_damage)
    return True, cooldown_max, attack_damage, attacker_vel * speed_penalty


## 观测向量

构建结构化观测向量，24 维，所有字段归一化到 [0,1] 或 [-1,1]。

```
自身状态 (8 维):
  [0:2]   player.pos / FIELD
  [2:4]   player.vel / max_speed
  [4]     player.hp / player.max_hp
  [5:7]   player 到边界的最近距离
  [7]     player.attack_cooldown_remaining / player.attack_cooldown

敌方相对状态 (8 维):
  [8:10]  (bot.pos - player.pos) / FIELD
  [10:12] (bot.vel - player.vel) / max_speed
  [12]    bot.hp / bot.max_hp
  [13:15] bot 到边界的最近距离
  [15]    bot.attack_cooldown_remaining / bot.attack_cooldown

距离特征 (2 维):
  [16]    distance / 场地对角线
  [17]    (distance - attack_range) / 场地对角线

时间特征 (1 维):
  [18]    step_count / max_steps

预留 (5 维):
  [19:24] 0
```


In [22]:
OBS_DIM = 24

# 环境常量
MOVE_ACTIONS      = ["STOP", "W", "A", "S", "D", "WA", "WD", "SA", "SD"]
N_SKILLS          = 2
ATTACK_RANGE      = 45.0
ATTACK_DAMAGE     = 10.0
ATTACK_COOLDOWN   = 30
PLAYER_MAX_SPEED  = 6.0
BOT_MAX_SPEED     = 5.0
MAX_STEPS         = 3600   # 60fps * 60s = 1 分钟

def build_obs(player: Player, bot: Bot, step_count: int, max_steps: int) -> np.ndarray:
    field_scale     = np.array([FIELD_W, FIELD_H], dtype=np.float32)
    field_diag      = np.sqrt(FIELD_W**2 + FIELD_H**2)
    max_speed_scale = np.array([max(PLAYER_MAX_SPEED, BOT_MAX_SPEED)] * 2, dtype=np.float32)

    self_pos_norm       = player.pos / field_scale
    self_vel_norm       = player.vel / max_speed_scale
    self_hp_norm        = np.array([player.hp / player.max_hp], dtype=np.float32)
    self_boundary       = np.array([
        min(player.pos[1], FIELD_H - player.pos[1]) / FIELD_H,
        min(player.pos[0], FIELD_W - player.pos[0]) / FIELD_W,
    ], dtype=np.float32)
    self_cooldown       = np.array([player.attack_cooldown_remaining / max(player.attack_cooldown, 1)],
                                   dtype=np.float32)

    enemy_rel_pos_norm  = (bot.pos - player.pos) / field_scale
    enemy_rel_vel_norm  = (bot.vel - player.vel) / max_speed_scale
    enemy_hp_norm       = np.array([bot.hp / bot.max_hp], dtype=np.float32)
    enemy_boundary      = np.array([
        min(bot.pos[1], FIELD_H - bot.pos[1]) / FIELD_H,
        min(bot.pos[0], FIELD_W - bot.pos[0]) / FIELD_W,
    ], dtype=np.float32)
    enemy_cooldown      = np.array([bot.attack_cooldown_remaining / max(bot.attack_cooldown, 1)],
                                   dtype=np.float32)

    dist = np.linalg.norm(player.pos - bot.pos)
    dist_norm   = np.array([dist / field_diag], dtype=np.float32)
    dist_range  = np.array([(dist - ATTACK_RANGE) / field_diag], dtype=np.float32)

    time_norm = np.array([step_count / max_steps], dtype=np.float32)

    obs = np.concatenate([
        self_pos_norm, self_vel_norm, self_hp_norm, self_boundary, self_cooldown,
        enemy_rel_pos_norm, enemy_rel_vel_norm, enemy_hp_norm, enemy_boundary, enemy_cooldown,
        dist_norm, dist_range,
        time_norm,
        np.zeros(5, dtype=np.float32),
    ])
    return obs.astype(np.float32)

## 奖励配置系统

奖励由外部权重向量驱动。环境在 `info` 字典中提供原始事件，`compute_reward()` 根据权重配置计算标量奖励。

| 分项              | 事件来源         | 默认含义           |
|-------------------|------------------|--------------------|
| `damage_dealt`    | info["damage_dealt"]   | 对 Bot 造成的伤害  |
| `damage_taken`    | info["damage_taken"]   | 自身受到的伤害    |
| `survival_bonus`  | 每帧固定值              | 存活奖励          |
| `distance_penalty`| 玩家与 Bot 的距离      | 距离惩罚/奖励     |
| `kill_bonus`      | done 时 Bot 死亡        | 击杀奖励          |


In [23]:
def compute_reward(
    info: dict, step_count: int,
    player: Player, bot: Bot,
    weights: dict | None = None,
) -> float:
    """根据权重配置计算标量奖励。"""
    if weights is None:
        weights = {
            "damage_dealt":     1.0,
            "damage_taken":    -1.0,
            "survival_bonus":   0.01,
            "distance_penalty": 0.0,
            "kill_bonus":       5.0,
        }

    reward = 0.0
    reward += info.get("damage_dealt", 0.0) * weights.get("damage_dealt", 0.0)
    reward += info.get("damage_taken", 0.0) * weights.get("damage_taken", 0.0)

    if "survival_bonus" in weights:
        reward += weights["survival_bonus"]

    if "distance_penalty" in weights and abs(weights["distance_penalty"]) > 1e-8:
        dist = np.linalg.norm(player.pos - bot.pos)
        field_diag = np.sqrt(FIELD_W**2 + FIELD_H**2)
        dist_norm = dist / field_diag
        reward += weights["distance_penalty"] * dist_norm

    reward += info.get("kill_bonus", 0.0) * weights.get("kill_bonus", 0.0)
    return reward

## ArenaEnv

### reset()
- 随机化双方出生位置（彼此距离 >= 200px）
- 重置 hp、速度、冷却计数器
- 返回 `obs`（观测向量，24维）

### step(action)
- `action`: `(move_idx, skill0_bool, skill1_bool)` — 来自 RL 模型
- 执行一帧物理更新（玩家 + Bot 分别 update）
- 处理攻击判定、伤害、死亡
- 返回 `(obs, reward, done, info)`

### render()
- 复用现有 pygame 绘制逻辑，新增 Bot 方块(红色)、hp 条

### close()
- pygame.quit()


In [24]:
class ArenaEnv:
    def __init__(self, render_mode: str | None = None, reward_weights: dict | None = None):
        self.render_mode = render_mode
        self.screen = None
        self.clock = None
        self.font = None
        self.step_count = 0

        self.reward_weights = reward_weights


    def _init_pygame(self):
        pygame.font.init()
        self.screen = pygame.display.set_mode((WINDOW_W, WINDOW_H))
        pygame.display.set_caption("RL Arena - 1v1")
        self.clock = pygame.time.Clock()
        self.font = pygame.font.Font(None, 18)

    def reset(self) -> np.ndarray:
        if self.render_mode == "human" and self.screen is None:
            self._init_pygame()
        self.step_count = 0

        while True:
            p_pos = np.array([
                np.random.uniform(50, FIELD_W - 50),
                np.random.uniform(50, FIELD_H - 50),
            ], dtype=np.float32)
            b_pos = np.array([
                np.random.uniform(50, FIELD_W - 50),
                np.random.uniform(50, FIELD_H - 50),
            ], dtype=np.float32)
            if np.linalg.norm(p_pos - b_pos) >= 200.0:
                break

        self.player = Player(
            pos=p_pos,
            vel=np.array([0.0, 0.0], dtype=np.float32),
            size=20, color=COLOR_PLAYER,
            accel=0.8, friction=0.90, max_speed=PLAYER_MAX_SPEED,
            hp=100.0, max_hp=100.0,
            attack_range=ATTACK_RANGE, attack_damage=ATTACK_DAMAGE,
            attack_cooldown=ATTACK_COOLDOWN, attack_cooldown_remaining=0,
        )
        self.bot = Bot(
            pos=b_pos,
            vel=np.array([0.0, 0.0], dtype=np.float32),
            size=20, color=(255, 80, 80),
            accel=0.6, friction=0.90, max_speed=BOT_MAX_SPEED,
            hp=100.0, max_hp=100.0,
            attack_range=ATTACK_RANGE, attack_damage=ATTACK_DAMAGE,
            attack_cooldown=ATTACK_COOLDOWN, attack_cooldown_remaining=0,
        )

        return build_obs(self.player, self.bot, self.step_count, MAX_STEPS)

    def step(self, action: tuple) -> tuple:
        """
        action: (move_idx, skill0_bool, skill1_bool)
        返回: (obs, reward, done, info)
        """
        move_idx, skill0, skill1 = action
        self.step_count += 1

        player_accel = action_to_accel(move_idx, self.player.accel)
        self.player.pos, self.player.vel = physics_update_entity(
            self.player.pos, self.player.vel, player_accel,
            self.player.friction, self.player.max_speed, self.player.size,
        )

        bot_accel = self.bot.get_action(self.player.pos)
        self.bot.pos, self.bot.vel = physics_update_entity(
            self.bot.pos, self.bot.vel, bot_accel,
            self.bot.friction, self.bot.max_speed, self.bot.size,
        )

        info = {"player_attack": False, "bot_attack": False,
                "player_hp": self.player.hp, "bot_hp": self.bot.hp}

        player_attacked = skill0 or skill1
        if player_attacked:
            attacked, new_cd, dmg, new_vel = try_attack(
                self.player.pos, self.bot,
                self.player.attack_range, self.player.attack_damage,
                self.player.attack_cooldown_remaining, self.player.attack_cooldown,
                self.player.vel,
            )
            if attacked:
                self.player.attack_cooldown_remaining = new_cd
                self.player.vel = new_vel
                info["player_attack"] = True
                info["damage_dealt"] = info.get("damage_dealt", 0.0) + dmg

        if self.bot.should_attack(self.player.pos):
            attacked, new_cd, dmg, new_vel = try_attack(
                self.bot.pos, self.player,
                self.bot.attack_range, self.bot.attack_damage,
                self.bot.attack_cooldown_remaining, self.bot.attack_cooldown,
                self.bot.vel,
            )
            if attacked:
                self.bot.attack_cooldown_remaining = new_cd
                self.bot.vel = new_vel
                info["bot_attack"] = True
                info["damage_taken"] = info.get("damage_taken", 0.0) + dmg

        if self.player.attack_cooldown_remaining > 0:
            self.player.attack_cooldown_remaining -= 1
        if self.bot.attack_cooldown_remaining > 0:
            self.bot.attack_cooldown_remaining -= 1

        reward = compute_reward(info, self.step_count, self.player, self.bot, self.reward_weights)

        done = False
        if self.player.is_dead():
            done = True
            info["winner"] = "bot"
        elif self.bot.is_dead():
            done = True
            info["winner"] = "player"
            info["kill_bonus"] = 1.0
        elif self.step_count >= MAX_STEPS:
            done = True
            info["winner"] = "player" if self.player.hp > self.bot.hp else \
                             "bot" if self.bot.hp > self.player.hp else "draw"

        obs = build_obs(self.player, self.bot, self.step_count, MAX_STEPS)
        return obs, reward, done, info

    def render(self):
        if self.render_mode != "human" or self.screen is None:
            return

        self.screen.fill(COLOR_BG)

        for x in range(0, FIELD_W + 1, 50):
            sx = FIELD_OFFSET[0] + x
            pygame.draw.line(self.screen, COLOR_GRID, (sx, FIELD_OFFSET[1]),
                             (sx, FIELD_OFFSET[1] + FIELD_H), 1)
        for y in range(0, FIELD_H + 1, 50):
            sy = FIELD_OFFSET[1] + y
            pygame.draw.line(self.screen, COLOR_GRID, (FIELD_OFFSET[0], sy),
                             (FIELD_OFFSET[0] + FIELD_W, sy), 1)

        border_rect = pygame.Rect(FIELD_OFFSET[0], FIELD_OFFSET[1], FIELD_W, FIELD_H)
        pygame.draw.rect(self.screen, COLOR_BORDER, border_rect, 2)

        pygame.draw.rect(self.screen, self.player.color, self.player.rect)
        pygame.draw.rect(self.screen, self.bot.color, self.bot.rect)

        for entity, line_color in [(self.player, (255, 255, 100)), (self.bot, (255, 255, 100))]:
            cx = int(FIELD_OFFSET[0] + entity.pos[0])
            cy = int(FIELD_OFFSET[1] + entity.pos[1])
            vx = int(entity.vel[0] * 10)
            vy = int(entity.vel[1] * 10)
            pygame.draw.line(self.screen, line_color, (cx, cy), (cx+vx, cy+vy), 2)

        for entity, is_player in [(self.player, True), (self.bot, False)]:
            bar_w = 40
            bar_h = 6
            bar_x = int(FIELD_OFFSET[0] + entity.pos[0] - bar_w / 2)
            bar_y = int(FIELD_OFFSET[1] + entity.pos[1] - entity.size / 2 - 12)
            hp_ratio = entity.hp / entity.max_hp
            pygame.draw.rect(self.screen, (60, 60, 60),
                             (bar_x, bar_y, bar_w, bar_h))
            hp_color = (80, 220, 80) if is_player else (220, 80, 80)
            pygame.draw.rect(self.screen, hp_color,
                             (bar_x, bar_y, int(bar_w * hp_ratio), bar_h))

        hud = [
            f"Step: {self.step_count}/{MAX_STEPS}",
            f"Player HP: {self.player.hp:.0f}  Bot HP: {self.bot.hp:.0f}",
            f"Player CD: {self.player.attack_cooldown_remaining}",
        ]
        for i, line in enumerate(hud):
            text = self.font.render(line, True, COLOR_HUD)
            self.screen.blit(text, (10, 10 + i * 18))

        pygame.display.flip()
        self.clock.tick(FPS)

    def close(self):
        try:
            if self.screen is not None:
                pygame.quit()
                self.screen = None
        except Exception:
            pass


## 环境集成测试


In [25]:
def test_env_random(render: bool = False, n_episodes: int = 5):
    env = None
    try:
        env = ArenaEnv(render_mode="human" if render else None)

        for ep in range(n_episodes):
            obs = env.reset()
            total_reward = 0.0
            done = False
            steps = 0

            while not done:
                move_idx = np.random.randint(0, 9)
                skill0 = np.random.randint(0, 2)
                skill1 = np.random.randint(0, 2)
                obs, reward, done, info = env.step((move_idx, skill0, skill1))
                total_reward += reward
                steps += 1

                if render:
                    env.render()

            print(f"Episode {ep+1}: steps={steps}, reward={total_reward:.2f}, "
                  f"winner={info.get('winner','?')}, "
                  f"player_hp={info.get('player_hp',100):.0f}, "
                  f"bot_hp={info.get('bot_hp',100):.0f}")

        print("All tests passed.")
    except Exception as e:
        print(f"Test failed: {e}")
    finally:
        if env is not None:
            env.close()


# test_env_random(render=False, n_episodes=10)

test_env_random(render=True, n_episodes=1)

Episode 1: steps=376, reward=3.76, winner=bot, player_hp=10, bot_hp=10
All tests passed.


## 模块2：PPO训练管线

### 网络架构
- **Actor**: 共享MLP -> move_head(9-discrete, Categorical) + skill_head(2×Bernoulli via sigmoid)
- **Critic**: 共享MLP -> value_head(1-scalar)

### 算法参数
| 参数 | 值 |
|------|-----|
| clip_epsilon | 0.2 |
| n_epochs | 10 |
| rollout_steps | 2048 |
| gamma | 0.99 |
| gae_lambda | 0.95 |
| lr | 3e-4 |
| batch_size | 128 |
| entropy_coef | 0.01 |
| value_coef | 0.5 |

### 训练方式
1. 收集 rollout (2048 步环境交互)
2. 计算 GAE 优势 + 优势归一化
3. 对 rollout 数据做 10 个 epoch 的 mini-batch 更新
4. 重复直到达到目标步数


In [ ]:
from ppo_trainer import Actor, Critic, PPOTrainer, plot_metrics, run_training

# 快速验证训练流程 (约 10000 步, 约 5 次迭代)
env = ArenaEnv(render_mode=None)
metrics = run_training(
    env,
    total_steps=10240,
    rollout_steps=2048,
    hidden_dim=128,
    lr=3e-4,
    log_interval=1,
)
env.close()

plot_metrics(metrics, title="PPO Training (10K steps)")

In [ ]:
# 完整训练并保存模型 (约 100K 步)
# env = ArenaEnv(render_mode=None)
# metrics = run_training(
#     env,
#     total_steps=100_000,
#     rollout_steps=2048,
#     hidden_dim=128,
#     lr=3e-4,
#     save_path="model_default.pth",
#     log_interval=1,
# )
# env.close()
# plot_metrics(metrics, title="PPO Training (100K steps)")

## 模块4：行为差异验证

### 核心假设
玩家通过自定义奖励函数，能塑造出行为差异化的智能体。

### 实验设计
用两组极端奖励权重分别训练，对比行为指标。

| 配置 | damage_dealt | damage_taken | survival | distance | kill |
|------|-------------|-------------|----------|----------|------|
| 激进型 | +2.0 | -0.1 | 0 | 0 | +10.0 |
| 保守型 | +0.5 | -5.0 | +1.0 | -0.5 | +2.0 |

### 验证指标
- 平均每局存活步数
- 平均每局造成/受到伤害
- 平均与敌方距离
- 平均每局攻击次数
- 胜率
- 训练曲线收敛性

### 通过标准
两组在关键指标上有统计显著差异（p < 0.05），行为模式肉眼可辨识。


In [ ]:
from ppo_trainer import (
    run_training, evaluate_agent, compare_agents,
    load_actor_for_eval, run_behavior_experiment,
)

# 快速验证：少量步数训练两组并对比
agg_summary, con_summary, p_vals = run_behavior_experiment(
    ArenaEnv,
    total_steps=20480,    # 快速测试 ~10 iterations
    hidden_dim=128,
    eval_episodes=50,     # 50 局评估
)

In [ ]:
# 完整实验：100K 步训练 + 100 局评估
# agg_summary, con_summary, p_vals = run_behavior_experiment(
#     ArenaEnv,
#     total_steps=100_000,
#     hidden_dim=128,
#     eval_episodes=100,
# )